In [1]:
import os, json, sys
from typing import Dict, Any, List
from neo4j import GraphDatabase
import openai


In [2]:

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "Thesis*1234"
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

In [3]:
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
client = openai.OpenAI(api_key=OPENAI_API_KEY)

In [4]:
def init_constraints():
    """
    Ensure an id-based uniqueness for Entity nodes so MERGE works deterministically.
    """
    cypher = """
    CREATE CONSTRAINT entity_id_unique IF NOT EXISTS
    FOR (e:Entity) REQUIRE e.id IS UNIQUE
    """
    with driver.session() as session:
        session.run(cypher)

# --------- LLM extraction ----------
SYSTEM_PROMPT = """You are an information extraction model.
Given a single sentence or short passage, you will produce a compact knowledge graph as JSON with this exact schema:

{
  "nodes": [
    {
      "id": "string (unique within this output, e.g., n1, n2, ...)",
      "name": "string (canonical name)",
      "type": "string (entity type like Person, Organization, Product, Location, Event, Concept, etc.)",
      "properties": { "optionalKey": "optionalValue", "...": "..." }
    }
  ],
  "relationships": [
    {
      "source": "string (node id)",
      "target": "string (node id)",
      "type": "string (relationship type, e.g., WORKS_AT, FOUNDED, LOCATED_IN, PURCHASED, PART_OF, MENTIONS, etc.)",
      "properties": { "optionalKey": "optionalValue" }
    }
  ]
}

Rules:
- Return ONLY a valid JSON object matching the schema. No prose.
- Keep it minimal but meaningful. Avoid duplicate nodes.
- Use stable, short ids like n1, n2, n3.
- If nothing can be extracted, return {"nodes": [], "relationships": []}.
"""

def extract_graph(user_text: str) -> Dict[str, Any]:
    """
    Uses the OpenAI Chat Completions API with JSON mode to produce a nodes/relationships structure.
    """
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
        ],
    )
    content = resp.choices[0].message.content
    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        # Fallback to an empty graph if something unexpected happens
        data = {"nodes": [], "relationships": []}

    # Light validation
    data.setdefault("nodes", [])
    data.setdefault("relationships", [])
    # Ensure required fields
    for n in data["nodes"]:
        n.setdefault("properties", {})
    for r in data["relationships"]:
        r.setdefault("properties", {})
    return data

# --------- Neo4j upsert ----------
NODE_UPSERT = """
UNWIND $nodes AS n
MERGE (e:Entity {id: n.id})
SET   e.name = n.name,
      e.type = n.type
SET   e += coalesce(n.properties, {})
"""

# We use a generic relationship type with a 'type' property to avoid requiring APOC for dynamic rel types
REL_UPSERT = """
UNWIND $rels AS r
MATCH (s:Entity {id: r.source})
MATCH (t:Entity {id: r.target})
MERGE (s)-[rel:RELATED_TO {type: r.type}]->(t)
SET   rel += coalesce(r.properties, {})
"""

def upsert_graph(graph: Dict[str, Any]):
    nodes: List[Dict[str, Any]] = graph.get("nodes", [])
    rels: List[Dict[str, Any]] = graph.get("relationships", [])
    if not nodes and not rels:
        return

    with driver.session() as session:
        if nodes:
            session.run(NODE_UPSERT, parameters={"nodes": nodes})
        if rels:
            # Filter out relationships that reference non-existent nodes (defensive)
            node_ids = {n["id"] for n in nodes} | {
                # also include any pre-existing ids? (we can't cheaply know), so we just pass through.
            }
            session.run(REL_UPSERT, parameters={"rels": rels})

# --------- Orchestration ----------
def sentence_to_neo4j(user_sentence: str) -> Dict[str, Any]:
    init_constraints()
    graph = extract_graph(user_sentence)
    upsert_graph(graph)
    return graph

In [5]:
text="The fourth season of Chicago Fire, an American drama television series with executive producer Dick Wolf, and producers Derek Haas, Michael Brandt, and Matt Olmstead, was ordered on February 5, 2015, by NBC,[1] and premiered on October 13, 2015 and concluded on May 17, 2016.[2] The season contained 23 episodes.[3]"
if not text:
    print("No text provided.")
    sys.exit(0)

result = sentence_to_neo4j(text)
print(json.dumps(result, indent=2))

ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ()):
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 111] Connection refused)